In [0]:
# RevOps CRM Pipeline Project
catalog = "flora_workspace"
schema = "default"
volume = "revops_crm_data"
file_name = "sales_pipeline.csv"

# Unity Catalog paths

path_volume = f"/Volumes/{catalog}/{schema}/{volume}"

path_table = f"{catalog}.{schema}"

In [0]:
print("Catalog:", catalog)
print("Schema:", schema)
print("Volume:", volume)
print("File:", file_name)
print("Volume Path:", path_volume)

Catalog: flora_workspace
Schema: default
Volume: revops_crm_data
File: sales_pipeline.csv
Volume Path: /Volumes/flora_workspace/default/revops_crm_data


In [0]:
%sql
CREATE VOLUME IF NOT EXISTS flora_workspace.default.revops_crm_data;

In [0]:
# Check files in the RevOps data volume

display(dbutils.fs.ls(path_volume))

path,name,size,modificationTime
dbfs:/Volumes/flora_workspace/default/revops_crm_data/accounts.csv,accounts.csv,4670,1788564679000
dbfs:/Volumes/flora_workspace/default/revops_crm_data/products.csv,products.csv,171,1788564679000
dbfs:/Volumes/flora_workspace/default/revops_crm_data/sales_pipeline.csv,sales_pipeline.csv,637773,1788564680000
dbfs:/Volumes/flora_workspace/default/revops_crm_data/sales_teams.csv,sales_teams.csv,1284,1788564679000


In [0]:
%sql
SHOW CATALOGS;

catalog
flora_workspace
samples
system


In [0]:
%sql
SHOW SCHEMAS IN flora_workspace;

databaseName
default
information_schema


In [0]:
%sql
SHOW VOLUMES IN flora_workspace.default;

database,volume_name
default,revops_crm_data


In [0]:
# ==========================================
# REVOPS CRM PIPELINE & FORECAST PROJECT
# ==========================================

catalog = "flora_workspace"
schema = "default"
volume = "revops_crm_data"

file_name = "sales_pipeline.csv"

path_volume = f"/Volumes/{catalog}/{schema}/{volume}"

print("Volume Path:", path_volume)

Volume Path: /Volumes/flora_workspace/default/revops_crm_data


In [0]:
display(dbutils.fs.ls(path_volume))

path,name,size,modificationTime
dbfs:/Volumes/flora_workspace/default/revops_crm_data/accounts.csv,accounts.csv,4670,1788564679000
dbfs:/Volumes/flora_workspace/default/revops_crm_data/products.csv,products.csv,171,1788564679000
dbfs:/Volumes/flora_workspace/default/revops_crm_data/sales_pipeline.csv,sales_pipeline.csv,637773,1788564680000
dbfs:/Volumes/flora_workspace/default/revops_crm_data/sales_teams.csv,sales_teams.csv,1284,1788564679000


In [0]:

df_sales_pipeline = spark.read.csv(
    f"{path_volume}/sales_pipeline.csv",
    header=True,
    inferSchema=True
)

display(df_sales_pipeline)

opportunity_id,sales_agent,product,account,deal_stage,engage_date,close_date,close_value
1C1I7A6R,Moses Frase,GTX Plus Basic,Cancity,Won,2016-10-20,2017-03-01,1054
Z063OYW0,Darcel Schlecht,GTXPro,Isdom,Won,2016-10-25,2017-03-11,4514
EC4QE1BX,Darcel Schlecht,MG Special,Cancity,Won,2016-10-25,2017-03-07,50
MV1LWRNH,Moses Frase,GTX Basic,Codehow,Won,2016-10-25,2017-03-09,588
PE84CX4O,Zane Levy,GTX Basic,Hatfan,Won,2016-10-25,2017-03-02,517
ZNBS69V1,Anna Snelling,MG Special,Ron-tech,Won,2016-10-29,2017-03-01,49
9ME3374G,Vicki Laflamme,MG Special,J-Texon,Won,2016-10-30,2017-03-02,57
7GN8Q4LL,Markita Hansen,GTX Basic,Cheers,Won,2016-11-01,2017-03-07,601
OLK9LKZB,Niesha Huffines,GTX Plus Basic,Zumgoity,Won,2016-11-01,2017-03-03,1026
HAXMC4IX,James Ascencio,MG Advanced,null,Engaging,2016-11-03,null,null


In [0]:
# Check column names and data types

df_sales_pipeline.printSchema()

root
 |-- opportunity_id: string (nullable = true)
 |-- sales_agent: string (nullable = true)
 |-- product: string (nullable = true)
 |-- account: string (nullable = true)
 |-- deal_stage: string (nullable = true)
 |-- engage_date: date (nullable = true)
 |-- close_date: date (nullable = true)
 |-- close_value: integer (nullable = true)



In [0]:
# Count sales opportunities

df_sales_pipeline.count()

8800

In [0]:
# Check null values in each column

from pyspark.sql import functions as F

df_sales_pipeline.select([
    F.sum(F.col(column).isNull().cast("int")).alias(column)
    for column in df_sales_pipeline.columns
]).display()

opportunity_id,sales_agent,product,account,deal_stage,engage_date,close_date,close_value
0,0,0,1425,0,500,2089,2089


In [0]:
# Show the columns available for analysis

print(df_sales_pipeline.columns)

['opportunity_id', 'sales_agent', 'product', 'account', 'deal_stage', 'engage_date', 'close_date', 'close_value']


In [0]:
# Reload the CRM sales pipeline dataset

df_sales_pipeline = spark.read.csv(
    "/Volumes/flora_workspace/default/revops_crm_data/sales_pipeline.csv",
    header=True,
    inferSchema=True
)

print("Rows:", df_sales_pipeline.count())
print("Columns:", df_sales_pipeline.columns)

Rows: 8800
Columns: ['opportunity_id', 'sales_agent', 'product', 'account', 'deal_stage', 'engage_date', 'close_date', 'close_value']


In [0]:
df_sales_pipeline.printSchema()

root
 |-- opportunity_id: string (nullable = true)
 |-- sales_agent: string (nullable = true)
 |-- product: string (nullable = true)
 |-- account: string (nullable = true)
 |-- deal_stage: string (nullable = true)
 |-- engage_date: date (nullable = true)
 |-- close_date: date (nullable = true)
 |-- close_value: integer (nullable = true)



In [0]:
df_sales_pipeline.printSchema()

root
 |-- opportunity_id: string (nullable = true)
 |-- sales_agent: string (nullable = true)
 |-- product: string (nullable = true)
 |-- account: string (nullable = true)
 |-- deal_stage: string (nullable = true)
 |-- engage_date: date (nullable = true)
 |-- close_date: date (nullable = true)
 |-- close_value: integer (nullable = true)



In [0]:
# Create a permanent Delta table from the PySpark DataFrame

table_name = "flora_workspace.default.sales_pipeline"

df_sales_pipeline.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(table_name)

print(f"Permanent table created successfully: {table_name}")

Permanent table created successfully: flora_workspace.default.sales_pipeline


In [0]:
# Load the CRM sales pipeline dataset

df_sales_pipeline = spark.read.csv(
    "/Volumes/flora_workspace/default/revops_crm_data/sales_pipeline.csv",
    header=True,
    inferSchema=True
)

print("Rows:", df_sales_pipeline.count())
print("Columns:", df_sales_pipeline.columns)

Rows: 8800
Columns: ['opportunity_id', 'sales_agent', 'product', 'account', 'deal_stage', 'engage_date', 'close_date', 'close_value']


In [0]:
# Make the Python DataFrame available to SQL

df_sales_pipeline.createOrReplaceTempView("sales_pipeline_raw")

print("SQL view created successfully")

SQL view created successfully


In [0]:
%sql
SELECT
    deal_stage,
    COUNT(*) AS opportunity_count
FROM flora_workspace.default.sales_pipeline
GROUP BY deal_stage
ORDER BY opportunity_count DESC;

deal_stage,opportunity_count
Won,4238
Lost,2473
Engaging,1589
Prospecting,500


In [0]:
%sql
SELECT
    sales_agent,
    COUNT(*) AS won_opportunities,
     CONCAT('$', FORMAT_NUMBER(SUM(close_value), 0)) AS amount
     FROM flora_workspace.default.sales_pipeline
WHERE deal_stage = 'Won'
GROUP BY sales_agent
ORDER BY amount DESC;

sales_agent,won_opportunities,amount
Vicki Laflamme,221,"$478,396"
Kary Hendrixson,209,"$454,298"
Cassey Cress,163,"$450,489"
Donn Cantrell,158,"$445,860"
Reed Clapper,155,"$438,336"
Zane Levy,161,"$430,068"
Corliss Cosme,150,"$421,036"
James Ascencio,135,"$413,533"
Daniell Hammack,114,"$364,229"
Maureen Marcano,149,"$350,395"


In [0]:
%sql
SELECT
    product,
    COUNT(*) AS opportunity_count,
     CONCAT('$', FORMAT_NUMBER(SUM(close_value), 0)) AS amount
FROM flora_workspace.default.sales_pipeline
GROUP BY product
ORDER BY amount DESC;

product,opportunity_count,amount
GTX Plus Basic,1383,"$705,275"
GTX Basic,1866,"$499,263"
MG Special,1651,"$43,768"
GTK 500,40,"$400,612"
GTXPro,1480,"$3,510,578"
GTX Plus Pro,968,"$2,629,651"
MG Advanced,1412,"$2,216,387"


In [0]:
%sql


SELECT
    CONCAT(
        '$',
        FORMAT_NUMBER(
            AVG(close_value),
            0
        )
    ) AS average_won_deal_size

FROM flora_workspace.default.sales_pipeline

WHERE deal_stage = 'Won'
  AND close_value IS NOT NULL;


average_won_deal_size
"$2,361"


In [0]:
%sql
SELECT
    deal_stage,
    COUNT(*) AS opportunity_count
FROM flora_workspace.default.sales_pipeline
GROUP BY deal_stage
HAVING deal_stage not in ('Won','Lost')

deal_stage,opportunity_count
Engaging,1589
Prospecting,500


In [0]:
%sql

WITH account_opportunities AS (
    SELECT
        account,
        COUNT(*) AS open_opportunity_count
    FROM flora_workspace.default.sales_pipeline
    WHERE deal_stage NOT IN ('Won', 'Lost')
      AND account IS NOT NULL
    GROUP BY account
),

total_open AS (
    SELECT
        SUM(open_opportunity_count) AS total_open_opportunities
    FROM account_opportunities
)

SELECT
    account,
    open_opportunity_count,

    CONCAT(
        ROUND(
            100.0 * open_opportunity_count / total_open_opportunities,
            2
        ),
        '%'
    ) AS opportunity_share_pct,

    RANK() OVER (
        ORDER BY open_opportunity_count DESC
    ) AS opportunity_rank

FROM account_opportunities
CROSS JOIN total_open

ORDER BY opportunity_rank;

account,open_opportunity_count,opportunity_share_pct,opportunity_rank
Donquadtech,14,2.11%,1
Betasoloin,14,2.11%,1
Inity,13,1.96%,3
Iselectrics,13,1.96%,3
Blackzim,12,1.81%,5
Faxquote,12,1.81%,5
Plussunin,12,1.81%,5
Zoomit,12,1.81%,5
Dontechi,12,1.81%,5
Condax,11,1.66%,10


In [0]:
%sql

WITH rep_opportunities AS (
    SELECT
        sales_agent,
        COUNT(*) AS open_opportunity_count
    FROM flora_workspace.default.sales_pipeline
    WHERE deal_stage NOT IN ('Won', 'Lost')
      AND sales_agent IS NOT NULL
    GROUP BY sales_agent
),

total_open AS (
    SELECT
        SUM(open_opportunity_count) AS total_open_opportunities
    FROM rep_opportunities
)

SELECT
    sales_agent,
    open_opportunity_count,

    CONCAT(
        ROUND(
            100.0 * open_opportunity_count / total_open_opportunities,
            2
        ),
        '%'
    ) AS opportunity_share_pct,

    RANK() OVER (
        ORDER BY open_opportunity_count DESC
    ) AS opportunity_rank

FROM rep_opportunities
CROSS JOIN total_open

ORDER BY opportunity_rank;

sales_agent,open_opportunity_count,opportunity_share_pct,opportunity_rank
Darcel Schlecht,194,9.29%,1
Anna Snelling,112,5.36%,2
Vicki Laflamme,104,4.98%,3
Kary Hendrixson,103,4.93%,4
Versie Hillebrand,97,4.64%,5
Kami Bicknell,90,4.31%,6
Zane Levy,88,4.21%,7
Marty Freudenburg,87,4.16%,8
Cassey Cress,85,4.07%,9
Gladys Colclough,85,4.07%,9


In [0]:
%sql

SELECT
    sales_agent,

    COUNT(*) AS won_opportunities,

    ROUND(
        AVG(
            DATEDIFF(close_date, engage_date)
        ),
        1
    ) AS avg_sales_cycle_days

FROM flora_workspace.default.sales_pipeline

WHERE deal_stage = 'Won'
  AND engage_date IS NOT NULL
  AND close_date IS NOT NULL

GROUP BY sales_agent

ORDER BY avg_sales_cycle_days DESC;

sales_agent,won_opportunities,avg_sales_cycle_days
Moses Frase,129,64.7
Lajuana Vencill,127,62.9
Violet Mclelland,122,58.0
Wilburn Farren,55,56.5
Markita Hansen,130,55.9
Niesha Huffines,105,54.6
Maureen Marcano,149,54.1
Rosalina Dieter,72,54.0
Vicki Laflamme,221,53.6
James Ascencio,135,53.4
